# Observability with Langsmith
Lets test Langsmith observability using a non-trivial parallel chain from a previous notebook. This chain returns the answer, a summary and the sentiment of the answer.

![Langchain workflow](images/parallel-chain.png)



In [ ]:
%pip install -qU langchain-ollama langchain-core langchain-community --quiet
%pip install langsmith --quiet

Import Langsmith variables

In [ ]:
# Load dotenv
import os
from dotenv import load_dotenv
load_dotenv()
# LANGCHAIN_API_KEY=
# LANGCHAIN_TRACING_V2=true
# LANGCHAIN_ENDPOINT=https://api.smith.langchain.com
# LANGCHAIN_PROJECT=langsmith-observe1

True

Import libraries

In [10]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

## Main model

In [11]:
llm = ChatOllama(
    model="gemma4:e4b",
    temperature=0.1,
)

In [12]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant"),
    ("user", "{input}")
    ])

## Summary Chain

In [13]:
summary_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant that makes great summaries, Given the text, generate a concise summary."),
    ("user", "{input}")
    ])

In [14]:
summary_chain = summary_prompt | llm | StrOutputParser()

## Sentiment chain

In [16]:
sentiment_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant that specializes on sentiment analysis. \
     Given a text, classify it as positive, neutral or negative. Return a single word with your classification. \
     No formatting or special characters."),
    ("user", "{input}")
    ])

In [17]:
sentiment_chain = sentiment_prompt | llm | StrOutputParser()

## Parallel chain

In [18]:
#    answer=RunnablePassthrough(),
parallel_chain = RunnableParallel (
    answer=RunnableLambda(lambda x: x["input"]),
    summary=summary_chain,
    sentiment=sentiment_chain,
)

## Main chain
Concatenate the LLM with the parallel chain

In [19]:
#preprocessor = RunnableLambda(lower_case) | RunnableLambda(mask_credit_card_numbers)
main_chain = prompt | llm | {"input": StrOutputParser()} | parallel_chain

Testing the main chain

In [20]:
# This is a positive one
response = main_chain.invoke({"input": "I am so happy today. By the way what is the colour of the sky?"})

In [21]:
import pprint
pprint.pprint(response)

{'answer': "I'm so happy to hear that you're having a wonderful day! 😊\n"
           '\n'
           'As for the color of the sky, the most common answer is **blue**.\n'
           '\n'
           "However, the sky's color is actually magical and changes "
           'constantly depending on a few things:\n'
           '\n'
           '1.  **Time of Day:**\n'
           "    *   **Midday:** It's usually a bright, clear blue because of "
           "how Earth's atmosphere scatters sunlight (this is called Rayleigh "
           'scattering).\n'
           '    *   **Sunrise/Sunset:** It often turns brilliant shades of '
           '**orange, pink, red, and yellow** because the sunlight has to '
           'travel through more of the atmosphere, scattering away the blue '
           'light.\n'
           '    *   **Overcast/Stormy:** It can look **gray** or dark.\n'
           '\n'
           '2.  **Weather:**\n'
           '    *   If there are clouds, the sky will reflect the color of t

LangChain's underlying Runnables automatically check for variables like ```LANGCHAIN_API_KEY``` in the environment. When they are present, they wrap the execution call in a trace payload that is automatically sent to LangSmith.

The run above generated the following trace in Langsmith:

![Langsmith run](images/langsmith-run1.png)

In [22]:
# So hard to get negative thoughts from the LLM :D
response = main_chain.invoke({"input": "I feel kind of sad. Can you give me 3 negative thoughts. Please, don't try to be conforting"})

In [23]:
pprint.pprint(response)

{'answer': '1. Nothing you do will ever be enough.\n'
           '2. You are fundamentally misunderstood by everyone who matters.\n'
           '3. This feeling of sadness is permanent and will never lift.',
 'sentiment': 'Negative',
 'summary': 'This text conveys a profound sense of hopelessness, deep '
            'inadequacy, and isolation, suggesting that the speaker feels '
            'fundamentally misunderstood by important people and believes '
            'their sadness is permanent.'}


This second generated the following trace in Langsmith:

![Langsmith second run](images/langsmith-run2.png)

The waterfall view in Langsmith shows the timeline of events. In particular it shows that the two branches of the "RunnableParallel" indeed run in parallel

![Langsmith second run](images/langsmith-run2-waterfall.png)